# Portfolio Risk Engine Example

This notebook is the living example for the project. As new features are added, we can extend this notebook with new sections instead of creating disconnected demos.


In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'bigdata').is_dir() and (candidate / 'README.md').exists():
            return candidate
    raise RuntimeError('Could not locate the project root directory.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root added to sys.path: {PROJECT_ROOT}')

Project root added to sys.path: /Users/jiangzhenhao/Desktop/bigdata


## 1. Universe Definition

The current version uses a multi-asset ETF universe across equities, fixed income, commodities, and an FX proxy.

In [2]:
from bigdata.universe import ASSET_LIST, ASSET_CLASS_MAPPING, ASSET_UNIVERSE

print('Asset list:')
print(ASSET_LIST)

print('\nAsset class mapping:')
print(ASSET_CLASS_MAPPING)

ASSET_UNIVERSE

Asset list:
['SPY', 'QQQ', 'IWM', 'TLT', 'IEF', 'GLD', 'USO', 'UUP', 'EEM']

Asset class mapping:
{'SPY': 'Equities', 'QQQ': 'Equities', 'IWM': 'Equities', 'TLT': 'Fixed Income', 'IEF': 'Fixed Income', 'GLD': 'Commodities', 'USO': 'Commodities', 'UUP': 'FX Proxy', 'EEM': 'Equities'}


[{'ticker': 'SPY', 'asset_class': 'Equities'},
 {'ticker': 'QQQ', 'asset_class': 'Equities'},
 {'ticker': 'IWM', 'asset_class': 'Equities'},
 {'ticker': 'TLT', 'asset_class': 'Fixed Income'},
 {'ticker': 'IEF', 'asset_class': 'Fixed Income'},
 {'ticker': 'GLD', 'asset_class': 'Commodities'},
 {'ticker': 'USO', 'asset_class': 'Commodities'},
 {'ticker': 'UUP', 'asset_class': 'FX Proxy'},
 {'ticker': 'EEM', 'asset_class': 'Equities'}]

## 2. Data Pipeline Example

This section runs the end-to-end data pipeline:

- download daily adjusted close prices
- align to a common business-day calendar
- forward fill internal gaps
- compute daily simple returns
- create rolling train/test windows


In [3]:
from bigdata.data_pipeline import build_data_pipeline

pipeline_result = build_data_pipeline(
    tickers=ASSET_LIST,
    start_date='2020-01-01',
    missing_ratio_threshold=0.10,
    winsorize_limit=0.01,
    train_window=252,
    test_window=21,
    step_size=21,
)

pipeline_result

DataPipelineResult(clean_return_matrix=Ticker           EEM       GLD       IEF       IWM       QQQ       SPY  \
2020-01-03 -0.018567  0.013269  0.006683 -0.003921 -0.009160 -0.007572   
2020-01-06 -0.002448  0.010490 -0.001076  0.001332  0.006443  0.003815   
2020-01-07 -0.000670  0.003935 -0.001437 -0.003326 -0.000139 -0.002812   
2020-01-08  0.005805 -0.007502 -0.002338  0.003095  0.007516  0.005330   
2020-01-09  0.006659 -0.005652  0.000722  0.001210  0.008474  0.006780   
...              ...       ...       ...       ...       ...       ...   
2026-03-26 -0.033960 -0.031455 -0.008075 -0.017393 -0.023868 -0.017859   
2026-03-27 -0.004868  0.030027  0.000106 -0.017540 -0.019537 -0.017052   
2026-03-30 -0.008152 -0.000289  0.007082 -0.014356 -0.007643 -0.003343   
2026-03-31  0.033420  0.030027  0.001784  0.035015  0.033854  0.029068   
2026-04-01  0.007748  0.017500 -0.004191  0.006290  0.012353  0.007534   

Ticker           TLT       USO       UUP  
2020-01-03  0.015401  0.02888

In [4]:
pipeline_result.clean_price_matrix.tail()

Ticker,EEM,GLD,IEF,IWM,QQQ,SPY,TLT,USO,UUP
2026-03-26,55.470001,400.640015,94.589996,247.440002,573.789978,645.090027,86.110001,117.260002,27.809999
2026-03-27,55.200001,414.700012,94.599998,243.100006,562.580017,634.090027,85.639999,124.199997,27.840000
2026-03-30,54.750000,414.579987,95.269997,239.610001,558.280029,631.969971,86.779999,129.830002,27.980000
2026-03-31,56.790001,430.290009,95.440002,248.000000,577.179993,650.340027,86.690002,127.250000,27.780001
2026-04-01,57.230000,437.820007,95.040001,249.559998,584.309998,655.239990,86.260002,124.089996,27.730000


In [5]:
pipeline_result.clean_return_matrix.tail()

Ticker,EEM,GLD,IEF,IWM,QQQ,SPY,TLT,USO,UUP
2026-03-26,-0.033960,-0.031455,-0.008075,-0.017393,-0.023868,-0.017859,-0.008406,0.034130,0.003971
2026-03-27,-0.004868,0.030027,0.000106,-0.017540,-0.019537,-0.017052,-0.005458,0.059185,0.001079
2026-03-30,-0.008152,-0.000289,0.007082,-0.014356,-0.007643,-0.003343,0.013312,0.045330,0.005029
2026-03-31,0.033420,0.030027,0.001784,0.035015,0.033854,0.029068,-0.001037,-0.019872,-0.007148
2026-04-01,0.007748,0.017500,-0.004191,0.006290,0.012353,0.007534,-0.004960,-0.024833,-0.001800


In [6]:
pipeline_result.missing_ratio_by_asset

Ticker
EEM    0.03681
GLD    0.03681
IEF    0.03681
IWM    0.03681
QQQ    0.03681
SPY    0.03681
TLT    0.03681
USO    0.03681
UUP    0.03681
dtype: float64

In [7]:
pipeline_result.dropped_assets

[]

In [8]:
first_window = pipeline_result.training_test_windows[0]

print('Train window:', first_window.train_start, '->', first_window.train_end)
print('Test window:', first_window.test_start, '->', first_window.test_end)
print('\nTrain shape:', first_window.train_data.shape)
print('Test shape:', first_window.test_data.shape)

Train window: 2020-01-03 00:00:00 -> 2020-12-21 00:00:00
Test window: 2020-12-22 00:00:00 -> 2021-01-19 00:00:00

Train shape: (252, 9)
Test shape: (21, 9)


## 3. Future Sections

Planned extensions for this notebook:

- risk metrics
- covariance estimation
- VaR / CVaR examples
- factor modeling
- rolling backtests
- stress testing
